# 06 • Basic RAG สำหรับคำถามงานคลังยาง

เป้าหมาย: ค้น SOP ที่เกี่ยวข้อง แล้วให้ LLM ตอบเป็นภาษาไทยพร้อมรหัสอ้างอิง เปรียบเทียบกับคำตอบที่ไม่มีเอกสาร

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

บริบทงานอ้างอิงจาก [เว็บไซต์กลุ่มศรีตรัง](https://www.sritranggroup.com/th/home) ส่วนรูปแบบ notebook อ้างอิงจาก [ตัวอย่าง GitHub](https://github.com/biodatlab/cpf-genai-workshop/tree/main/tutorial_notebooks) และใช้ [OpenAI API documentation](https://developers.openai.com/api/docs) สำหรับ syntax

## เตรียมพร้อมก่อนเริ่ม
เพิ่ม Secret ชื่อ `OPENAI_API_KEY` ใน Colab ด้วยตนเอง และเปิด **Notebook access** สำหรับ notebook นี้ โค้ดจะอ่านผ่าน `userdata.get()` โดยไม่แสดงค่า key

รันเซลล์จากบนลงล่าง หลังติดตั้ง package และเตรียม key แล้ว ทดสอบตัวอย่างหลักใน Colab ด้วย API แล้ว แต่ผล LLM อาจเปลี่ยนได้ การผ่านตัวอย่างนี้ไม่ใช่การรับรองทุกคำถามหรือการใช้งานจริง

ใช้ `gpt-4.1-mini` สำหรับสร้างคำตอบ (ไม่เกิน 500 tokens ต่อ request) และ `text-embedding-3-small` สำหรับ Embedding โดย Agent Loop จำกัดไว้ 5 รอบ

In [1]:
%pip -q install openai numpy

In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

In [3]:
chunks = [
    "[SOP-1 รับน้ำยาง] บันทึก Lot ผู้ขาย น้ำหนักสุทธิ ทะเบียนรถ และเวลารับเข้า เก็บรหัสผู้ขายเพื่อเชื่อมโยงข้อมูล Traceability",
    "[SOP-2 QC] เก็บผล DRC พร้อมรหัสตัวอย่างและ Lot หากผลยังไม่ครบ ให้ระบุสถานะรอ QC ห้ามสรุปว่าผ่านเกณฑ์จากข้อมูลที่หายไป",
    "[SOP-3 คลังสินค้า] Lot ที่รอ QC ต้องแยกในพื้นที่พักรอผลและติดป้าย Lot ตรวจสถานะก่อนยืนยันการจัดส่ง",
    "[SOP-4 ประสานงาน] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 30 ซม. ให้ประสานหัวหน้าคลังและฝ่ายขนส่งเพื่อตรวจสอบแผนรับส่งสินค้า",
    "[SOP-5 น้ำท่วม] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 50 ซม. ให้พักแผนโหลดสินค้าของพื้นที่ที่ได้รับผลกระทบ แจ้งหัวหน้าคลัง ฝ่ายขนส่ง และ EHS เพื่อประเมินตามแผนของพื้นที่",
    "[SOP-6 ส่งมอบ] เมื่อเวลารับส่งเปลี่ยน ให้ฝ่ายขนส่งประสานฝ่ายขาย แจ้งกำหนดการใหม่ที่ยืนยันแล้ว ห้ามรับรองเวลาส่งมอบที่ยังไม่ยืนยัน",
    "[SOP-7 ซ่อมบำรุง] เมื่อพบปัญหาเครื่องจักร ให้บันทึกอาการ รหัสเครื่อง และเวลา แล้วส่งใบงานให้ผู้รับผิดชอบซ่อมบำรุง",
    "[SOP-8 สิ่งแวดล้อม] หากพบข้อร้องเรียนด้านกลิ่นหรือน้ำเสีย ให้บันทึกเวลา จุดที่พบ และรายละเอียด ส่งต่อผู้รับผิดชอบสิ่งแวดล้อมเพื่อตรวจสอบ"
]

In [4]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts):
    result = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([item.embedding for item in result.data])

vectors = embed(chunks)
print("ขนาด Embedding matrix:", vectors.shape)

ขนาด Embedding matrix: (8, 1536)


In [5]:
def retrieve(question, k=2):
    query = embed([question])[0]
    scores = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query))
    indices = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in indices]

## คำตอบเมื่อยังไม่มี SOP
สังเกตว่า LLM อาจให้คำแนะนำทั่วไปที่ไม่ตรงกับเอกสารตัวอย่าง

In [6]:
question = "คลังยางแท่ง A มีระดับน้ำในสถานการณ์สมมติ 55 ซม. ต้องประสานงานและจัดการการส่งมอบอย่างไรตาม SOP?"
print(client.responses.create(
    model=MODEL, instructions="ตอบเป็นภาษาไทย ระบุข้อจำกัดของข้อมูล", input=question, max_output_tokens=500,
).output_text)

เนื่องจากข้อมูลที่ให้มาเป็นเพียงระดับน้ำในคลังยางแท่ง A ที่ 55 ซม. แต่ไม่ได้ระบุรายละเอียดเกี่ยวกับ SOP การส่งมอบยางแท่ง เช่น ขั้นตอนการตรวจสอบคุณภาพ การบรรจุภัณฑ์ การจัดเก็บ หรือการประสานงานกับฝ่ายอื่น ๆ ทำให้ไม่สามารถระบุแนวทางการจัดการและประสานงานได้อย่างชัดเจน

ข้อจำกัดของข้อมูล:  
- ไม่ทราบรายละเอียด SOP ที่ใช้กับคลังยางแท่ง A  
- ไม่มีข้อมูลเกี่ยวกับมาตรฐานการตรวจสอบคุณภาพและปริมาณ  
- ไม่มีข้อมูลฝ่ายที่เกี่ยวข้องและช่องทางการประสานงาน  

หากต้องการคำแนะนำที่ชัดเจน กรุณาให้ข้อมูลเกี่ยวกับ SOP ของการส่งมอบยางแท่ง เช่น ขั้นตอนการตรวจเช็คระดับน้ำ ปริมาณที่อนุญาตให้ส่งมอบ หรือผู้รับผิดชอบในแต่ละขั้นตอนเพิ่มเติมครับ


## คำตอบจาก SOP ที่ค้นพบ
อ่าน chunks ก่อนดูคำตอบ ตรวจว่าข้อความรองรับคำตอบจริง

In [7]:
question = "คลังยางแท่ง A มีระดับน้ำในสถานการณ์สมมติ 55 ซม. ต้องประสานงานและจัดการการส่งมอบอย่างไรตาม SOP?"
hits = retrieve(question)
for text, score in hits:
    print(round(score, 3), text)

context = "\n".join(text for text, score in hits)
response = client.responses.create(
    model=MODEL,
    instructions="ตอบเป็นภาษาไทยโดยใช้เฉพาะ SOP ที่ให้ อ้างรหัส SOP หากข้อมูลไม่เพียงพอให้ระบุว่าไม่พบข้อมูล ห้ามสร้างขั้นตอนเพิ่ม",
    input=f"บริบท:\n{context}\n\nคำถาม: {question}",
    max_output_tokens=500,
)
print(response.output_text)

0.63 [SOP-4 ประสานงาน] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 30 ซม. ให้ประสานหัวหน้าคลังและฝ่ายขนส่งเพื่อตรวจสอบแผนรับส่งสินค้า
0.482 [SOP-5 น้ำท่วม] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 50 ซม. ให้พักแผนโหลดสินค้าของพื้นที่ที่ได้รับผลกระทบ แจ้งหัวหน้าคลัง ฝ่ายขนส่ง และ EHS เพื่อประเมินตามแผนของพื้นที่
ตาม SOP ที่ให้ไว้

ระดับน้ำ 55 ซม. อยู่ในเกณฑ์ตั้งแต่ 50 ซม. ขึ้นไป

- ตาม SOP-5 น้ำท่วม ให้พักแผนโหลดสินค้าของพื้นที่ที่ได้รับผลกระทบ
- แจ้งหัวหน้าคลัง ฝ่ายขนส่ง และ EHS เพื่อประเมินตามแผนของพื้นที่

ดังนั้น คลังยางแท่ง A ต้องพักแผนโหลดสินค้าของพื้นที่ และแจ้งหัวหน้าคลัง ฝ่ายขนส่ง และ EHS เพื่อประเมินสถานการณ์ตาม SOP-5


## ลองทำด้วยตนเอง
1. เปรียบเทียบคำตอบก่อนและหลังให้ SOP
2. เปลี่ยน k แล้วดูว่า SOP ใดถูกส่งให้ LLM
3. ถามเรื่องค่าชดเชยสินค้า ตรวจว่าโมเดลระบุว่าเอกสารไม่มีข้อมูล